<h1>LD Pipeline 2024 (Version 1.0, PROD)</h1>
<p>
Dies ist die optimierte Version der Pipeline, die auf dem Code von Redlink basiert. Die folgenden Optimierungen wurden zusätzlich implementiert:
<ul>
    <li>Erstellung temporärer Datenbanktabellen zur Beschleunigung der Iterationen</li>
    <li>Batch-Verarbeitung der Triples im Arbeitsspeicher</li>
    <li>On-the-fly-Datenkomprimierung</li>
    <li>Verarbeitung der Triple-Dateien in einem einzigen Prozess (Inbox-, Temp- und Done-Ordner)</li>
    <li>Übermittlung von Optionsparametern (Batchgrössen, max. Anzahl Iterationen)</li>
</ul>
</p>

<h3>Lade die Umgebung</h3>

In [1]:
import os
import requests

try:
    initialized
except NameError:
    initialized = False

if not initialized:
    initialized = True
    os.chdir('../')
    import main
    import utils

<h3>Generiere die Triple-Dateien</h3>

In [ ]:
names = [ 'code', 'cube', 'groupCode', 'hierarchy', 'legalFoundation',
         'measureUnit', 'measure', 'property', 'room', 'time' ]
#names = ['observation']
names = ['time']

options = {
    'db_batch_size': 100000,
    'write_batch_size': 500000,
    'max_iteration': None
}

options = {
    'db_batch_size': 1,
    'write_batch_size': 1,
    'max_iteration': 1
}

for name in names:
    main.step(name=f"{name}Templating", env=main.Env.prod, options=options)

<h3>Setze den Fuseki-Server zurück (Optional)</h3>

In [ ]:
def reset_fuseki_server():
    server_url = 'http://localhost:8080/fuseki/ssz/update'
    query = """
        DELETE WHERE {
            ?s ?p ?o.
        }
    """
    response = requests.post(server_url, data={'update': query})
    if response.status_code == 200:
        print("All triples are successfully deleted.")
    else:
        print(f"Status code: {response.status_code}, Error: {response.text}")

reset_fuseki_server()

<h3>Übertrage die Dateien in der Inbox zu dem lokalen Fuseki-Server</h3>

In [ ]:
main.step(name="uploadToFuseki", env=main.Env.prod)

<h3>Übertrage die Dateien in der Inbox zu Stardog</h3>

In [ ]:
main.step(name="uploadToStardog", env=main.Env.prod)

<h3>SPARQL-Abfragen an Stardog senden</h3>

In [2]:
stardog_graph_uri = utils.get_stardog_graph_uri(env=main.Env.prod)
df = utils.execute_sparql(f"""
    SELECT * FROM <{stardog_graph_uri}> WHERE {{
        ?sub ?pred ?obj
    }} LIMIT 10
""", env=main.Env.prod)
display(df)

,sub,pred,obj
0,https://ld.stadt-zuerich.ch/statistics/000040,https://schema.org/workExample,https://ld.admin.ch/application/visualize
1,https://ld.stadt-zuerich.ch/statistics/000041,https://schema.org/workExample,https://ld.admin.ch/application/visualize
2,https://ld.stadt-zuerich.ch/statistics/000042,https://schema.org/workExample,https://ld.admin.ch/application/visualize
3,https://ld.stadt-zuerich.ch/statistics/000043,https://schema.org/workExample,https://ld.admin.ch/application/visualize
4,https://ld.stadt-zuerich.ch/statistics/000044,https://schema.org/workExample,https://ld.admin.ch/application/visualize
5,https://ld.stadt-zuerich.ch/statistics/000045,https://schema.org/workExample,https://ld.admin.ch/application/visualize
6,https://ld.stadt-zuerich.ch/statistics/000050,https://schema.org/workExample,https://ld.admin.ch/application/visualize
7,https://ld.stadt-zuerich.ch/statistics/000051,https://schema.org/workExample,https://ld.admin.ch/application/visualize
8,https://ld.stadt-zuerich.ch/statistics/000052,https://schema.org/workExample,https://ld.admin.ch/application/visualize
9,https://ld.stadt-zuerich.ch/statistics/000053,https://schema.org/workExample,https://ld.admin.ch/application/visualize
